## 1. Imports & backend

In [1]:
// Absolute file:
// URL bypasses Deno's package-resolution (the local
// `deno.json` import map isn't always visible to Deno from inside Jupyter).
// Adjust the path if you move the minml checkout.
import createMinml from "../build/minml_js.js";
console.log("imports OK");

imports OK


In [2]:
// Try GPU first; fall back to CPU/WASM. Switch to { backend: "cpu" } to force WASM.
const m = await createMinml();
await m.initWebGPU();
m.setDefaultDevice(m.Device.WebGPU);

## 2. Inspecting Tensors
Simple function for inspecting tensors in the notebook.minml feature requests:
1. Add a better function for this purpose into minml.  E.g., `await mytensor.display()` can be used in notebooks to get a pretty-print of the tensor, similar to the display logic for a `jnp.array`.2. Add a corresponding `tree.display` function (or something)What would be even better is if there is a built-in Typescript way to configure how an object is displayed.  If we overrode that on `Tensor`, then we'd get 2 for free.

In [16]:
const a = m.array([1, 2, 3, 4], m.Device.WebGPU);
const b = m.array([10, 20, 30, 40], m.Device.WebGPU);
const twos = m.array([2, 2, 2, 2],  m.Device.WebGPU);
const c = m.mul(m.add(a, b), twos);
const promise = c.tolist()
console.log(await promise);

[ 22, 44, 66, 88 ]


# Simple generative model
Code currently doesn't run -- will pause here, since I think this is already a useful small example of some code it would be nice to be able to run.

In [ ]:
class Trace {
    obs!: Tensor; // (N_obs,)  
    assignments!: Tensor; // (N_obs,)  
    dists!: Tensor; // (N_clusters, N_categories)  
    static simulate(key: PRNGKey, N_clusters: number, N_categories: number, N_obs: number): Trace {    
       const t = new Trace();    
       const [k1, k2, k3] = key.split(3);    
       t.dists = new Dirichlet(m, m.ones([N_categories])).sample(k1, [N_clusters]);    
       t.assignments = m.randint(k2, 0, N_clusters, [N_obs]);    
       const simulateOneObservation = (key: PRNGKey, idx: Tensor) =>      
           (new Categorical(m, t.dists.gather(idx))).sample(key, []);    
       t.obs = vmap(simulateOneObservation, [0, 0])(k3.split(N_obs), t.assignments);    
       return t;  
    }
}

Single trace simulation:

In [ ]:
const trace = Trace.simulate(PRNGKey.new(0), 3, 8);
({  
    dists: await inspect(trace.dists),  
    assignments: await inspect(trace.assignments),  
    obs: await inspect(trace.obs),
});

In [ ]:
const traces = vmap(Trace.simulate, [0, null, null, null])(    
    PRNGKey.new(0),    3,    4,    10)({    
    dists: await inspect(traces.dists),    
    assignments: await inspect(traces.assignments),    
    obs: await inspect(traces.obs),  
});